# RoboDawn 기준선 재현

Colab의 **L4 GPU 런타임**에서 순서대로 실행한다. 로컬에서 만든 `robodawn_reproduction.zip`을 첫 셀에 업로드한다.

수집한 500개 seed와 시연 entry를 기준으로 `demo_randomized`, 최대 45턴, 응답 예산 8000토큰, 명령 primer와 과제 시연 1개를 사용한다. Gemini 3.8 Flash는 OpenRouter를 거치지 않고 서비스 계정 키로 Vertex AI를 직접 호출한다. 성공률 실행은 반값인 Flex PayGo(`--tier flex`)로, 턴 지연 측정은 standard로 한다.

공개 하네스와 저장한 프롬프트는 일치한다. 요청 본문도 공개 하네스 그대로다(응답 8000토큰, 추론 필드 없이 모델 기본 추론). 단 Gemini 3.8 Flash가 무시하는 temperature 0은 보내지 않는다. 원 게이트웨이와 그 추론 강도는 공개되지 않아 재현의 한계로 남긴다.

첫 실행은 두 과제에서 1개 에피소드씩이다. 성공률 비교용 10개 에피소드 실행은 별도 셀에서 켠다. 설치 후 모델 호출에는 API 비용, GPU 사용에는 Colab 사용량이 든다.


In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile

uploaded = files.upload()
bundle_name = next(name for name in uploaded if name.endswith('.zip'))
with zipfile.ZipFile(bundle_name) as archive:
    for name in archive.namelist():
        target = (Path('/content') / name).resolve()
        if not name.startswith('branchlab/') or not target.is_relative_to('/content/branchlab'):
            raise ValueError('Unexpected archive path')
    archive.extractall('/content')
root = Path('/content/branchlab')
checksums = json.loads((root / 'bundle_checksums.json').read_text())
for name, expected in checksums.items():
    assert hashlib.sha256((root / name).read_bytes()).hexdigest() == expected, name
gpu_name = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True).strip()
if 'L4' not in gpu_name:
    raise RuntimeError('Select an L4 GPU runtime before installing')
print('Bundle verified. GPU:', gpu_name)


RoboTwin 환경과 무작위화 장면에 필요한 배경 질감을 설치한다. 설치 로그는 `/content/bootstrap.log`와 각 단계의 로그에 남는다.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
for name in ['bootstrap', 'download_assets', 'setup_env', 'install_rt']:
    shutil.copy2(root / f'colab/robotwin/{name}.sh', f'/content/{name}.sh')
install_env = dict(os.environ, WITH_TEXTURES='1')
with open('/content/bootstrap.log', 'w') as log:
    status = subprocess.run(['bash', '/content/bootstrap.sh'], cwd='/content', env=install_env,
                            stdout=log, stderr=subprocess.STDOUT)
print('\n'.join(Path('/content/bootstrap.log').read_text().splitlines()[-30:]))
status.check_returncode()
assert 'BOOTSTRAP_DONE' in Path('/content/bootstrap.log').read_text()
subprocess.run(['bash', str(root / 'colab/robotwin/prepare_robodawn.sh')], check=True)
python_rt = '/content/mamba/envs/rt/bin/python'


Vertex AI 서비스 계정 키(JSON)를 업로드하고 실행 조건을 확인한다. 키는 `/content/vertex_key.json`에만 두고 결과 묶음에 넣지 않으며, 마지막 셀에서 지운다. 호출 비용은 키의 GCP 프로젝트로 청구된다.

In [ ]:
uploaded_key = files.upload()
key_name = next(name for name in uploaded_key if name.endswith('.json'))
key_path = Path('/content/vertex_key.json')
key_path.write_bytes(uploaded_key[key_name])
key_path.chmod(0o600)
Path(key_name).unlink(missing_ok=True)
if json.loads(key_path.read_text()).get('type') != 'service_account':
    raise ValueError('Upload a service account key')
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = str(key_path)
runner = root / 'scripts/run_robodawn_baseline.py'
smoke_tasks = ['adjust_bottle', 'place_empty_cup']
for task in smoke_tasks:
    subprocess.run([python_rt, str(runner), '--task', task, '--dry-run'], check=True)

두 과제의 에피소드 0을 flex로 실행한다. 완료 후 seed·시연 entry·반환 모델·트래픽 종류(ON_DEMAND_FLEX)·추론 토큰을 점검한다. 점검에 실패해도 원 로그는 보존되며, 이 단계만으로 성공률 재현이 검증된 것은 아니다.

In [ ]:
for task in smoke_tasks:
    subprocess.run([python_rt, str(runner), '--task', task, '--episodes', '1', '--tier', 'flex'], check=True)

앞 단계의 조건 점검이 통과한 뒤 아래 값을 바꾼다. `RUN_TEN_EPISODES`는 과제별 10개 에피소드를 flex로 실행하고, `RUN_STANDARD_LATENCY`는 턴 지연 비교용으로 에피소드 0을 standard로 실행한다. flex 결과의 시간은 대기열 지연이 섞여 지연 비교에 쓰지 않는다. 같은 결과 폴더의 완료된 에피소드는 건너뛴다.

In [ ]:
RUN_TEN_EPISODES = False
RUN_STANDARD_LATENCY = False
for task in smoke_tasks:
    check = root / f'outputs/robodawn/gemini_flash_flex/{task}/shard_0/condition_check.json'
    if (RUN_TEN_EPISODES or RUN_STANDARD_LATENCY) and (not check.exists() or json.loads(check.read_text())['errors']):
        raise RuntimeError(f'First-episode condition check has not passed: {task}')
    if RUN_TEN_EPISODES:
        subprocess.run([python_rt, str(runner), '--task', task, '--episodes', '10', '--tier', 'flex'], check=True)
    if RUN_STANDARD_LATENCY:
        subprocess.run([python_rt, str(runner), '--task', task, '--episodes', '1', '--tier', 'standard'], check=True)
if not (RUN_TEN_EPISODES or RUN_STANDARD_LATENCY):
    print('Ten-episode and latency runs are disabled.')

결과와 로그를 내려받고 서비스 계정 키를 지운다. 작업을 마치면 Colab GPU 런타임을 종료한다.

In [ ]:
result_root = root / 'outputs/robodawn'
if result_root.exists():
    archive = shutil.make_archive('/content/robodawn_results', 'zip', root_dir=result_root)
    files.download(archive)
else:
    print('No run outputs yet.')
Path('/content/vertex_key.json').unlink(missing_ok=True)